# Chapter 2: Working with Text Data

This notebook covers the fundamentals of processing text data for LLM training:
- Text tokenization strategies
- Vocabulary building
- Dataset creation and batching
- Efficient data loading with PyTorch

## 2.1: Understanding Tokenization

In [ ]:
import tiktoken
import re
from collections import Counter
import numpy as np

# Initialize GPT-2 tokenizer
tokenizer = tiktoken.get_encoding("gpt2")
print(f"GPT-2 Tokenizer initialized")
print(f"Vocabulary size: {tokenizer.n_vocab}")

In [ ]:
# Example 1: Simple tokenization
text = "Hello, world!"
tokens = tokenizer.encode(text)
decoded = tokenizer.decode(tokens)

print(f"Original text: {text}")
print(f"Tokens: {tokens}")
print(f"Token count: {len(tokens)}")
print(f"Decoded: {decoded}")

In [ ]:
# Example 2: How tokenization affects different texts
texts = [
    "The quick brown fox",
    "Natural Language Processing",
    "Numbers: 12345",
    "Emoji: 🚀🤖💡",
    "Special chars: @#$%^&*()"
]

for text in texts:
    tokens = tokenizer.encode(text)
    token_ratio = len(tokens) / len(text.split())
    print(f"Text: '{text}'")
    print(f"  Tokens: {tokens}")
    print(f"  Word count: {len(text.split())}, Token count: {len(tokens)}")
    print()

## 2.2: Building a Dataset from Raw Text

In [ ]:
# Create a larger training corpus
training_data = """
Natural Language Processing is a subfield of linguistics, computer science, and artificial intelligence concerned with the interactions between computers and human language. NLP is used to apply machine learning algorithms to text and speech.

A language model is a probability distribution over sequences of words. Language models are useful for many applications, including machine translation, speech recognition, and text generation.

Transformer models have become the state-of-the-art architecture for many NLP tasks. They are based on the self-attention mechanism, which allows the model to attend to different parts of the input sequence.

Large language models like GPT-2, GPT-3, and BERT have achieved remarkable results across a wide range of NLP benchmarks. These models are trained on large amounts of text data from the internet.

The attention mechanism is a key component of transformer models. It allows the model to selectively focus on relevant parts of the input, which helps improve performance on downstream tasks.

Tokenization is the process of breaking text into smaller units called tokens. These tokens can be words, characters, or subword units like bytes or BPE tokens.

Pretraining on large amounts of unlabeled data has become a dominant approach in modern NLP. Models like BERT and GPT are first pretrained on a language modeling task, then fine-tuned on specific downstream tasks.

Transfer learning has been very successful in NLP. By pretraining on a large corpus and fine-tuning on task-specific data, we can achieve strong performance even with limited labeled data.
"" " * 3  # Repeat for more data

# Tokenize the entire dataset
tokens = tokenizer.encode(training_data)
print(f"Training data size: {len(training_data)} characters")
print(f"Token count: {len(tokens)} tokens")
print(f"Average tokens per character: {len(tokens) / len(training_data):.2f}")
print(f"\nFirst 100 tokens: {tokens[:100]}")

## 2.3: Creating Training Pairs (Input-Target)

In [ ]:
# For language modeling, we create sliding windows
# Input: tokens[i:i+context_length]
# Target: tokens[i+1:i+context_length+1]

context_length = 32

def create_training_pairs(tokens, context_length):
    """Create input-target pairs for language modeling"""
    pairs = []
    for i in range(len(tokens) - context_length):
        input_seq = tokens[i:i + context_length]
        target_seq = tokens[i + 1:i + context_length + 1]
        pairs.append((input_seq, target_seq))
    return pairs

pairs = create_training_pairs(tokens, context_length)
print(f"Total training pairs: {len(pairs)}")
print(f"\nExample pair:")
print(f"  Input tokens: {pairs[0][0][:10]}...")
print(f"  Target tokens: {pairs[0][1][:10]}...")
print(f"  Context length: {len(pairs[0][0])}")

In [ ]:
# Visualize the relationship between input and target
print("Input-Target relationship:")
print()

input_tokens = pairs[5][0]
target_tokens = pairs[5][1]

print("Position | Input Token | Target Token | Explanation")
print("-" * 60)

for i in range(min(10, len(input_tokens))):
    input_decoded = tokenizer.decode([input_tokens[i]])
    target_decoded = tokenizer.decode([target_tokens[i]])
    explanation = "predicts next" if i < len(input_tokens) - 1 else "last token"
    print(f"{i:8d} | {str(input_tokens[i]):11} | {str(target_tokens[i]):12} | {explanation}")

## 2.4: PyTorch Dataset and DataLoader

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class LanguageModelingDataset(Dataset):
    """Dataset for language modeling"""
    
    def __init__(self, tokens, context_length):
        self.tokens = tokens
        self.context_length = context_length
    
    def __len__(self):
        # Number of valid windows
        return len(self.tokens) - self.context_length
    
    def __getitem__(self, idx):
        # Input: current window
        input_ids = self.tokens[idx:idx + self.context_length]
        # Target: shifted window (predicting next token)
        target_ids = self.tokens[idx + 1:idx + self.context_length + 1]
        
        return (
            torch.tensor(input_ids, dtype=torch.long),
            torch.tensor(target_ids, dtype=torch.long)
        )

# Create dataset and dataloader
dataset = LanguageModelingDataset(tokens, context_length=128)
train_loader = DataLoader(dataset, batch_size=32, shuffle=True, drop_last=True)

print(f"Dataset size: {len(dataset)}")
print(f"Batch size: 32")
print(f"Number of batches: {len(train_loader)}")

In [ ]:
# Inspect a batch
batch_input, batch_target = next(iter(train_loader))

print(f"Batch input shape: {batch_input.shape}")
print(f"  (batch_size=32, context_length=128)")
print(f"\nBatch target shape: {batch_target.shape}")
print(f"  (batch_size=32, context_length=128)")

print(f"\nFirst sample in batch:")
print(f"  Input: {batch_input[0, :20].tolist()}")
print(f"  Target: {batch_target[0, :20].tolist()}")
print(f"\nNotice: target[i] = input[i+1] (shifted by 1)")

## 2.5: Data Statistics and Analysis

In [ ]:
# Analyze token distribution
from collections import Counter

token_counter = Counter(tokens)
most_common = token_counter.most_common(10)

print("Most common tokens:")
print("Rank | Token ID | Count | Decoded")
print("-" * 50)
for rank, (token_id, count) in enumerate(most_common, 1):
    decoded = tokenizer.decode([token_id])
    print(f"{rank:4d} | {token_id:8d} | {count:5d} | '{decoded}'")

In [ ]:
# Vocabulary statistics
unique_tokens = len(token_counter)
total_tokens = len(tokens)

print(f"Vocabulary Statistics:")
print(f"  Total tokens in corpus: {total_tokens:,}")
print(f"  Unique tokens: {unique_tokens:,}")
print(f"  Coverage of GPT-2 vocab: {unique_tokens / tokenizer.n_vocab * 100:.2f}%")

# Calculate token frequency statistics
counts = list(token_counter.values())
print(f"\nToken frequency statistics:")
print(f"  Max frequency: {max(counts)}")
print(f"  Min frequency: {min(counts)}")
print(f"  Average: {np.mean(counts):.2f}")
print(f"  Median: {np.median(counts):.2f}")

## 2.6: Handling Multiple Files and Large Datasets

In [ ]:
class StreamingDataset(Dataset):
    """Dataset that can handle streaming/large data"""
    
    def __init__(self, token_file, context_length, seed=42):
        """
        Args:
            token_file: Path or list of token sequences
            context_length: Length of input sequences
            seed: Random seed for reproducibility
        """
        self.token_file = token_file if isinstance(token_file, list) else [token_file]
        self.context_length = context_length
        self.seed = seed
    
    def __len__(self):
        # This is approximate for streaming datasets
        return 10000
    
    def __getitem__(self, idx):
        # In practice, this would load tokens lazily from disk
        # For now, using the tokens from memory
        token_idx = idx % (len(tokens) - self.context_length)
        
        input_ids = tokens[token_idx:token_idx + self.context_length]
        target_ids = tokens[token_idx + 1:token_idx + self.context_length + 1]
        
        return (
            torch.tensor(input_ids, dtype=torch.long),
            torch.tensor(target_ids, dtype=torch.long)
        )

print("StreamingDataset class defined for handling large datasets")

## 2.7: Data Augmentation Techniques

In [ ]:
class AugmentedLanguageDataset(Dataset):
    """Dataset with data augmentation for LLMs"""
    
    def __init__(self, tokens, context_length, noise_ratio=0.01):
        self.tokens = tokens
        self.context_length = context_length
        self.noise_ratio = noise_ratio
    
    def __len__(self):
        return len(self.tokens) - self.context_length
    
    def __getitem__(self, idx):
        input_ids = self.tokens[idx:idx + self.context_length].copy()
        target_ids = self.tokens[idx + 1:idx + self.context_length + 1].copy()
        
        # Optional: Add random noise to rare tokens
        # This helps model robustness
        if np.random.random() < self.noise_ratio:
            noise_pos = np.random.randint(0, len(input_ids))
            # Don't modify - just for demonstration
        
        return (
            torch.tensor(input_ids, dtype=torch.long),
            torch.tensor(target_ids, dtype=torch.long)
        )

print("Augmented dataset class defined")

## 2.8: Summary and Best Practices

### Key Takeaways:
1. **Tokenization** converts raw text into discrete tokens that models can process
2. **Context windows** create input-target pairs for self-supervised learning
3. **Dataloaders** efficiently batch and shuffle data during training
4. **Token analysis** helps understand data distribution and potential issues

### Best Practices:
- Use byte-pair encoding (BPE) or subword tokenization for better coverage
- Shuffle data during training for better generalization
- Use appropriate batch sizes based on GPU memory
- Monitor token distribution for data quality issues
- Use streaming datasets for very large corpora